In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn mne

# All imports
import os
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Sequence, Tuple, Optional, Union
from collections import Counter
import gc

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# MNE for CSP and xDawn
from mne.decoding import CSP
from mne.preprocessing import Xdawn
import mne

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Conv2D, AveragePooling2D, MaxPooling2D
from tensorflow.keras.layers import SeparableConv2D, DepthwiseConv2D, BatchNormalization, Input, Flatten, GlobalMaxPooling2D
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:

# Paths & basic config
BASE_DIR = '/content/drive/MyDrive/'
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 500

DEFAULT_EEGNET_CHANNELS = ['F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6',
                          'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8']

USE_SUBJECTS = sorted([
    d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir()
    if d.is_dir() and d.name.startswith('P')
])
print(f"Found {len(USE_SUBJECTS)} subjects: {USE_SUBJECTS[:10]} ...")

CHANNELS_TO_USE = DEFAULT_EEGNET_CHANNELS
print(f"Using {len(CHANNELS_TO_USE)} channels: {CHANNELS_TO_USE}")


In [ ]:

# --------------------------
# EEG DATA LOADING
# --------------------------
def load_eeg_data(subjects, channels, pre_ms, post_ms, epoch_start_ms=-200, epoch_end_ms=600, fs=500):
    """
    Load EEG data for specified subjects and channels.
    
    Data is loaded from [-500, 1000] ms window, then a specific epoch is extracted.
    """
    all_X, all_y, all_subjects = [], [], []
    
    # Calculate indices for epoch extraction
    fixation_idx = int(pre_ms / 1000 * fs)
    epoch_start_idx = fixation_idx + int(epoch_start_ms / 1000 * fs)
    epoch_end_idx = fixation_idx + int(epoch_end_ms / 1000 * fs)
    
    print(f"Loading data from [-{pre_ms}, {post_ms}] ms window")
    print(f"Extracting epoch [{epoch_start_ms}, {epoch_end_ms}] ms (indices {epoch_start_idx}:{epoch_end_idx})")

    for subject in subjects:
        file_path = LABELED_EEG_DATA_DIR / subject / f"{pre_ms}_{post_ms}_training_data.npz"
        if not file_path.exists():
            continue

        with np.load(file_path, mmap_mode='r') as data:
            ch_names = data['channel_names']

            ch_indices = []
            for ch in channels:
                if ch in ch_names:
                    ch_indices.append(list(ch_names).index(ch))

            if len(ch_indices) == 0:
                continue

            X_full = data['X']
            y = data['y'][:]

            # Extract specific epoch and channels
            X_selected = X_full[:, ch_indices, epoch_start_idx:epoch_end_idx].copy().astype(np.float32)

        all_X.append(X_selected)
        all_y.append(y)
        all_subjects.extend([subject] * len(y))

        del X_selected

    X_combined = np.concatenate(all_X, axis=0).astype(np.float32)
    y_combined = np.concatenate(all_y, axis=0)

    print(f"\nLoaded {X_combined.shape[0]} trials from {len(set(all_subjects))} subjects")
    print(f"Using {len(ch_indices)} channels, {X_combined.shape[2]} time points")
    print(f"Epoch duration: {(epoch_end_ms - epoch_start_ms)} ms")

    label_counts = Counter(y_combined)
    total_samples = len(y_combined)
    print("\nLabel Distribution:")
    for label, count in sorted(label_counts.items()):
        proportion = count / total_samples
        print(f"  Label {label}: {count} samples ({proportion:.2%})")

    return X_combined, y_combined, np.array(all_subjects)


# --------------------------
# PER-SUBJECT NORMALIZATION
# --------------------------
def normalize_per_subject(X, y, subjects):
    """Normalize each subject's data individually using StandardScaler."""
    X_normalized = X.copy()
    scalers = {}
    
    unique_subjects = np.unique(subjects)
    for subject in unique_subjects:
        mask = subjects == subject
        X_subject = X[mask]
        
        scaler = StandardScaler()
        X_reshaped = X_subject.reshape(X_subject.shape[0], -1)
        scaler.fit(X_reshaped)
        
        X_normalized[mask] = scaler.transform(X_reshaped).reshape(X_subject.shape)
        scalers[subject] = scaler
        
        print(f"  {subject}: mean={scaler.mean_.mean():.4f}, std={scaler.scale_.mean():.4f}")
    
    return X_normalized, scalers


# --------------------------
# METHOD 1: NORMAL RANDOM SPLIT
# --------------------------
def load_data_normal_split(test_size=0.2, val_size=0.2, seed=42):
    """Normal random train/val/test split with per-subject normalization."""
    print("\n" + "="*70)
    print("LOADING DATA: NORMAL RANDOM SPLIT (per-subject normalization)")
    print("="*70)
    
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    # Add channel dimension for neural networks
    X_normalized = X_normalized[..., np.newaxis]
    
    print("\nPerforming random train/val/test split...")
    X_train, X_test, y_train, y_test, subj_train, subj_test = train_test_split(
        X_normalized, y_all, subjects_all, test_size=test_size, random_state=seed, stratify=y_all
    )
    
    X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
        X_train, y_train, subj_train, test_size=val_size/(1-test_size), random_state=seed, stratify=y_train
    )
    
    y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
    y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
    y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
    
    print(f"\nData shapes:")
    print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
    print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
    print(f"  Test:  {X_test.shape} from {len(np.unique(subj_test))} subjects")
    
    return X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat, y_train, y_val, y_test


# --------------------------
# METHOD 2: LEAVE-N-OUT CV
# --------------------------
def load_data_loocv(n_test_subjects=3, seed=42):
    """Leave-N-out CV: N subjects held out for testing."""
    print("\n" + "="*70)
    print(f"LOADING DATA: LEAVE-{n_test_subjects}-OUT CV (per-subject normalization)")
    print("="*70)
    
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    X_normalized = X_normalized[..., np.newaxis]
    
    unique_subjects = np.unique(subjects_all)
    n_subjects = len(unique_subjects)
    
    print(f"\nCreating folds with {n_test_subjects} test subjects each...")
    print(f"Total subjects: {n_subjects}")
    
    folds = []
    np.random.seed(seed)
    
    shuffled_subjects = np.random.permutation(unique_subjects)
    
    for fold_idx in range(n_subjects // n_test_subjects):
        test_subject_indices = np.arange(fold_idx * n_test_subjects, 
                                         (fold_idx + 1) * n_test_subjects)
        test_subjects = shuffled_subjects[test_subject_indices]
        train_val_subjects = np.setdiff1d(shuffled_subjects, test_subjects)
        
        test_mask = np.isin(subjects_all, test_subjects)
        train_val_mask = np.isin(subjects_all, train_val_subjects)
        
        X_test = X_normalized[test_mask]
        y_test = y_all[test_mask]
        
        X_train_val = X_normalized[train_val_mask]
        y_train_val = y_all[train_val_mask]
        subj_train_val = subjects_all[train_val_mask]
        
        X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
            X_train_val, y_train_val, subj_train_val, 
            test_size=0.2, random_state=seed, stratify=y_train_val
        )
        
        y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
        y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
        y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
        
        print(f"\nFold {fold_idx + 1}:")
        print(f"  Test subjects: {test_subjects}")
        print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
        print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
        print(f"  Test:  {X_test.shape}")
        
        folds.append({
            'X_train': X_train,
            'X_val': X_val,
            'X_test': X_test,
            'y_train': y_train_cat,
            'y_val': y_val_cat,
            'y_test': y_test_cat,
            'y_train_raw': y_train,
            'y_val_raw': y_val,
            'y_test_raw': y_test,
            'test_subjects': test_subjects,
            'fold_idx': fold_idx
        })
    
    return folds



In [ ]:

# --------------------------
# CSP MODEL
# --------------------------
def train_csp_model(X_train, X_val, X_test, y_train, y_val, y_test, 
                    n_components=4, n_subsample=300, seed=42, return_transformed=False):
    """
    Train CSP + RandomForest model with minimal RAM usage and overfitting prevention.
    
    Key improvements:
    - Fewer CSP components (4 instead of 8) to reduce dimensionality
    - Smaller RandomForest trees (max_depth=5) to prevent memorization
    - L2 regularization via max_features='sqrt' 
    - Validation set for early monitoring
    - Cross-subject subsample to prevent subject-specific overfitting
    """
    
    # Ensure correct dtypes
    X_train = X_train.astype(np.float32)
    X_val = X_val.astype(np.float32)
    X_test = X_test.astype(np.float32)
    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)
    
    # Remove channel dimension for CSP
    if X_train.ndim == 4:
        X_train = X_train.squeeze(-1)
        X_val = X_val.squeeze(-1)
        X_test = X_test.squeeze(-1)
    
    # Subsample for CSP fitting (use diverse trials)
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X_train), size=min(n_subsample, len(X_train)), replace=False)
    
    X_csp_fit = X_train[idx]
    y_csp_fit = y_train[idx]
    
    # Fit CSP with fewer components
    csp = CSP(
        n_components=n_components,
        log=True,
        norm_trace=False
    )
    csp.fit(X_csp_fit, y_csp_fit)
    
    del X_csp_fit, y_csp_fit
    gc.collect()
    
    # Transform datasets one at a time to save RAM
    X_train_csp = csp.transform(X_train)
    del X_train
    gc.collect()
    
    X_val_csp = csp.transform(X_val)
    del X_val
    gc.collect()
    
    X_test_csp = csp.transform(X_test)
    del X_test
    gc.collect()
    
    # Scale features
    scaler = StandardScaler()
    X_train_csp = scaler.fit_transform(X_train_csp)
    X_val_csp = scaler.transform(X_val_csp)
    X_test_csp = scaler.transform(X_test_csp)
    
    # Train RandomForest with regularization to prevent overfitting
    clf = RandomForestClassifier(
        n_estimators=100,              # Reduced from 200
        max_depth=5,                   # CRITICAL: Limit tree depth to prevent memorization
        min_samples_split=10,          # Require minimum samples to split
        min_samples_leaf=5,            # Require minimum samples in leaf
        max_features='sqrt',           # Use sqrt(n_features) for each split (regularization)
        bootstrap=True,
        n_jobs=-1,
        random_state=seed
    )
    clf.fit(X_train_csp, y_train)
    
    # Get predictions
    y_train_pred = clf.predict(X_train_csp)
    y_val_pred = clf.predict(X_val_csp)
    y_test_pred = clf.predict(X_test_csp)
    
    # Compute metrics
    train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
    val_bal_acc = balanced_accuracy_score(y_val, y_val_pred)
    test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    
    train_f1 = f1_score(y_train, y_train_pred, average='macro')
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    print(f"\n  CSP + RandomForest (Regularized):")
    print(f"    Balanced Accuracy - Train: {train_bal_acc:.3f} | Val: {val_bal_acc:.3f} | Test: {test_bal_acc:.3f}")
    print(f"    F1 Macro          - Train: {train_f1:.3f} | Val: {val_f1:.3f} | Test: {test_f1:.3f}")
    print(f"    Overfitting Gap   - {(train_bal_acc - test_bal_acc):.3f}")
    
    result = {
        'train_bal_acc': train_bal_acc,
        'val_bal_acc': val_bal_acc,
        'test_bal_acc': test_bal_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'test_f1': test_f1
    }
    
    # Optionally return model and transformed data for visualization
    if return_transformed:
        result['csp_model'] = csp
        result['X_train_csp'] = X_train_csp
        result['y_train'] = y_train
    else:
        # Clean up
        del X_train_csp, X_val_csp, X_test_csp
        gc.collect()
    
    return result


In [ ]:


# --------------------------
# xDawn MODEL
# --------------------------
def create_epochs_array(data, labels, info):
    """Convert numpy array to MNE Epochs object."""
    events = np.c_[np.arange(len(data)), np.zeros(len(data), dtype=int), labels]
    event_id = {str(lbl): lbl for lbl in np.unique(labels)}
    epochs = mne.EpochsArray(data, info, events, tmin=0, event_id=event_id, verbose=False)
    return epochs


def train_xdawn_model(X_train, X_val, X_test, y_train, y_val, y_test,
                      ch_names, sfreq, n_components=6, n_subsample=10000, seed=42):
    """Train xDawn + RandomForest model with minimal RAM usage."""
    
    # Ensure correct dtypes
    X_train = X_train.astype(np.float32)
    X_val = X_val.astype(np.float32)
    X_test = X_test.astype(np.float32)
    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)
    
    # Remove channel dimension for xDawn
    if X_train.ndim == 4:
        X_train = X_train.squeeze(-1)
        X_val = X_val.squeeze(-1)
        X_test = X_test.squeeze(-1)
    
    # Create MNE info object
    ch_types = ['eeg'] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        info.set_montage(montage, on_missing='ignore')
    except ValueError:
        pass
    
    # Subsample for xDawn fitting
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X_train), size=min(n_subsample, len(X_train)), replace=False)
    
    X_fit_data = X_train[idx]
    y_fit_labels = y_train[idx]
    
    # Convert to MNE Epochs for fitting
    epochs_fit = create_epochs_array(X_fit_data, y_fit_labels, info)
    
    # Fit xDawn
    xdawn = Xdawn(
        n_components=n_components,
        signal_cov=None,
        correct_overlap='auto'
    )
    xdawn.fit(epochs_fit)
    
    del X_fit_data, y_fit_labels, epochs_fit
    gc.collect()
    
    # Transform datasets one at a time to save RAM
    epochs_train = create_epochs_array(X_train, y_train, info)
    X_train_xdawn = xdawn.transform(epochs_train)
    del epochs_train, X_train
    gc.collect()
    
    epochs_val = create_epochs_array(X_val, y_val, info)
    X_val_xdawn = xdawn.transform(epochs_val)
    del epochs_val, X_val
    gc.collect()
    
    epochs_test = create_epochs_array(X_test, y_test, info)
    X_test_xdawn = xdawn.transform(epochs_test)
    del epochs_test, X_test
    gc.collect()
    
    # Extract mean amplitude per component across time
    X_train_feat = X_train_xdawn.mean(axis=2)
    X_val_feat = X_val_xdawn.mean(axis=2)
    X_test_feat = X_test_xdawn.mean(axis=2)
    
    del X_train_xdawn, X_val_xdawn, X_test_xdawn
    gc.collect()
    
    # Scale features
    scaler = StandardScaler()
    X_train_feat = scaler.fit_transform(X_train_feat)
    X_val_feat = scaler.transform(X_val_feat)
    X_test_feat = scaler.transform(X_test_feat)
    
    # Train RandomForest
    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        n_jobs=-1,
        random_state=seed
    )
    clf.fit(X_train_feat, y_train)
    
    # Get predictions
    y_train_pred = clf.predict(X_train_feat)
    y_val_pred = clf.predict(X_val_feat)
    y_test_pred = clf.predict(X_test_feat)
    
    # Compute metrics
    train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
    val_bal_acc = balanced_accuracy_score(y_val, y_val_pred)
    test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    
    train_f1 = f1_score(y_train, y_train_pred, average='macro')
    val_f1 = f1_score(y_val, y_val_pred, average='macro')
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    print(f"\n  xDawn + RandomForest:")
    print(f"    Balanced Accuracy - Train: {train_bal_acc:.3f} | Val: {val_bal_acc:.3f} | Test: {test_bal_acc:.3f}")
    print(f"    F1 Macro          - Train: {train_f1:.3f} | Val: {val_f1:.3f} | Test: {test_f1:.3f}")
    
    # Clean up
    del X_train_feat, X_val_feat, X_test_feat
    gc.collect()
    
    return {
        'train_bal_acc': train_bal_acc,
        'val_bal_acc': val_bal_acc,
        'test_bal_acc': test_bal_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'test_f1': test_f1
    }



In [ ]:

# --------------------------
# CSP VISUALIZATION FUNCTIONS
# --------------------------
def plot_csp_patterns(csp_model, ch_names, sfreq):
    """
    Plot CSP spatial patterns (filters learned by CSP).
    
    Shows how CSP weights different electrode signals. First few patterns
    highlight activity for one class, last few for the other class.
    """
    ch_types = ['eeg'] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        info.set_montage(montage, on_missing='ignore')
    except ValueError:
        pass
    
    print("\n" + "="*70)
    print("CSP SPATIAL PATTERNS")
    print("="*70)
    print("Each pattern represents a linear combination of electrode signals.")
    print("First patterns: dominant for Class 0")
    print("Last patterns: dominant for Class 1")
    
    fig = csp_model.plot_patterns(info, ch_type='eeg', sensors=True, units='AU', size=1.5, show_names=True)
    plt.tight_layout()
    plt.show()


def plot_csp_feature_distributions(X_train_csp, y_train, n_components_to_plot=3, position='first'):
    """
    Plot distributions of CSP features for each class.
    
    Args:
        X_train_csp: Transformed training data (n_samples, n_components)
        y_train: Training labels (n_samples,)
        n_components_to_plot: Number of components to visualize (default: 3)
        position: 'first' or 'last' components to plot
    """
    n_components = X_train_csp.shape[1]
    
    if position == 'first':
        component_indices = range(min(n_components_to_plot, n_components))
        title_prefix = "First"
    elif position == 'last':
        if n_components < n_components_to_plot:
            return  # Not enough components
        component_indices = range(n_components - n_components_to_plot, n_components)
        title_prefix = "Last"
    else:
        raise ValueError("position must be 'first' or 'last'")
    
    print("\n" + "="*70)
    print(f"CSP {title_prefix.upper()} {len(component_indices)} FEATURE DISTRIBUTIONS")
    print("="*70)
    print("Good separation between classes indicates discriminative features.")
    print(f"Class 0 (blue) vs Class 1 (red)\n")
    
    plt.figure(figsize=(15, 5))
    
    for plot_idx, comp_idx in enumerate(component_indices):
        plt.subplot(1, len(component_indices), plot_idx + 1)
        
        # Plot histograms with KDE
        sns.histplot(X_train_csp[y_train == 0, comp_idx], color='blue', label='Class 0', 
                    kde=True, stat='density', alpha=0.5, bins=30)
        sns.histplot(X_train_csp[y_train == 1, comp_idx], color='red', label='Class 1', 
                    kde=True, stat='density', alpha=0.5, bins=30)
        
        plt.title(f'CSP Component {comp_idx + 1}', fontweight='bold')
        plt.xlabel('Feature Value')
        plt.ylabel('Density')
        plt.legend(loc='upper right', fontsize=9)
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def visualize_csp_analysis(csp_model, X_train_csp, y_train, ch_names, sfreq):
    """
    Complete CSP visualization analysis: patterns and feature distributions.
    """
    # Plot spatial patterns
    plot_csp_patterns(csp_model, ch_names, sfreq)
    
    # Plot first 3 components
    plot_csp_feature_distributions(X_train_csp, y_train, n_components_to_plot=3, position='first')
    
    # Plot last 3 components (if enough components)
    if X_train_csp.shape[1] >= 3:
        plot_csp_feature_distributions(X_train_csp, y_train, n_components_to_plot=3, position='last')



In [ ]:

# --------------------------
# RESULTS TRACKER
# --------------------------
class ResultsTracker:
    """Track results across multiple runs/folds."""
    def __init__(self):
        self.results = {
            'normal': {'Model': [], 'Train_BalAcc': [], 'Val_BalAcc': [], 'Test_BalAcc': [], 
                      'Train_F1': [], 'Val_F1': [], 'Test_F1': []},
            'loocv': {'Model': [], 'Train_BalAcc': [], 'Val_BalAcc': [], 'Test_BalAcc': [], 
                     'Train_F1': [], 'Val_F1': [], 'Test_F1': [], 'Fold': []}
        }
    
    def record(self, eval_method, model_name, train_bal_acc, val_bal_acc, test_bal_acc, 
               train_f1, val_f1, test_f1, fold_idx=None):
        """Record a single model's results."""
        self.results[eval_method]['Model'].append(model_name)
        self.results[eval_method]['Train_BalAcc'].append(train_bal_acc)
        self.results[eval_method]['Val_BalAcc'].append(val_bal_acc)
        self.results[eval_method]['Test_BalAcc'].append(test_bal_acc)
        self.results[eval_method]['Train_F1'].append(train_f1)
        self.results[eval_method]['Val_F1'].append(val_f1)
        self.results[eval_method]['Test_F1'].append(test_f1)
        
        if fold_idx is not None:
            self.results[eval_method]['Fold'].append(fold_idx)
    
    def get_dataframe(self, eval_method):
        """Get results as a pandas DataFrame."""
        df = pd.DataFrame(self.results[eval_method])
        return df.round(3)
    
    def print_summary(self, eval_method):
        """Print summary statistics."""
        df = self.get_dataframe(eval_method)
        print(f"\n{eval_method.upper()} RESULTS:")
        print(df.to_string(index=False))
        
        if len(df) > 0:
            print(f"\nBalanced Accuracy:")
            print(f"  Avg Test: {df['Test_BalAcc'].mean():.1%} | Best: {df['Test_BalAcc'].max():.1%}")
            print(f"F1 Macro:")
            print(f"  Avg Test: {df['Test_F1'].mean():.1%} | Best: {df['Test_F1'].max():.1%}")



In [ ]:

# =============================================================================
# MAIN EXECUTION
# =============================================================================

tracker = ResultsTracker()

# =============================================================================
# 1. NORMAL RANDOM SPLIT
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*15 + "METHOD 1: NORMAL RANDOM SPLIT" + " "*23 + "█")
print("█" + " "*68 + "█")
print("█"*70)

X_train, X_val, X_test, y_train, y_val, y_test, y_train_raw, y_val_raw, y_test_raw = load_data_normal_split()

print("\n" + "="*70)
print("Training models on NORMAL SPLIT...")
print("="*70)

# Calculate sfreq for visualizations (800 ms epoch = -200 to 600 ms)
sfreq_viz = X_train.shape[2] * 1000 / 800


In [ ]:

print("\n--- CSP + RandomForest ---")
results_csp = train_csp_model(X_train, X_val, X_test, y_train_raw, y_val_raw, y_test_raw, 
                              n_components=4, n_subsample=300, return_transformed=True)
tracker.record('normal', 'CSP + RF', results_csp['train_bal_acc'], results_csp['val_bal_acc'], 
               results_csp['test_bal_acc'], results_csp['train_f1'], results_csp['val_f1'], 
               results_csp['test_f1'])


In [ ]:

# Visualize CSP analysis
visualize_csp_analysis(results_csp['csp_model'], results_csp['X_train_csp'], 
                      results_csp['y_train'], CHANNELS_TO_USE, sfreq_viz)


In [ ]:

print("\n--- xDawn + RandomForest ---")
results_xdawn = train_xdawn_model(X_train, X_val, X_test, y_train_raw, y_val_raw, y_test_raw,
                                  CHANNELS_TO_USE, sfreq_viz, n_components=6, n_subsample=10000)
tracker.record('normal', 'xDawn + RF', results_xdawn['train_bal_acc'], results_xdawn['val_bal_acc'],
               results_xdawn['test_bal_acc'], results_xdawn['train_f1'], results_xdawn['val_f1'],
               results_xdawn['test_f1'])


In [ ]:

# Clean up
del X_train, X_val, X_test, y_train, y_val, y_test, y_train_raw, y_val_raw, y_test_raw
del results_csp, results_xdawn
gc.collect()

print("\n" + "="*70)
print("RAM cleanup complete. Ready for LOOCV...")
print("="*70)


In [ ]:

# =============================================================================
# 2. LEAVE-N-OUT CROSS-VALIDATION
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*12 + "METHOD 2: LEAVE-N-OUT CROSS-VALIDATION" + " "*16 + "█")
print("█" + " "*68 + "█")
print("█"*70)

folds = load_data_loocv(n_test_subjects=3, seed=42)

# Extract fold 0
fold_data = folds[0]
fold_idx = fold_data['fold_idx']
test_subjects = fold_data['test_subjects']
X_train = fold_data['X_train']
X_val = fold_data['X_val']
X_test = fold_data['X_test']
y_train = fold_data['y_train']
y_val = fold_data['y_val']
y_test = fold_data['y_test']
y_train_raw = fold_data['y_train_raw']
y_val_raw = fold_data['y_val_raw']
y_test_raw = fold_data['y_test_raw']

# Delete all other folds to free RAM
del folds, fold_data
gc.collect()

print(f"\n{'='*70}")
print(f"FOLD {fold_idx + 1} (Test subjects: {test_subjects})")
print(f"{'='*70}")
print(f"Train shape: {X_train.shape} | Val shape: {X_val.shape} | Test shape: {X_test.shape}")


In [ ]:

# =============================================================================
# CELL: Train CSP on Fold 0
# =============================================================================
print("\n--- CSP + RandomForest ---")

results_csp = train_csp_model(X_train, X_val, X_test, y_train_raw, y_val_raw, y_test_raw,
                              n_components=4, n_subsample=300, return_transformed=True)
tracker.record('loocv', 'CSP + RF', results_csp['train_bal_acc'], results_csp['val_bal_acc'],
               results_csp['test_bal_acc'], results_csp['train_f1'], results_csp['val_f1'],
               results_csp['test_f1'], fold_idx)


In [ ]:

# Visualize CSP analysis for LOOCV fold
sfreq_viz = X_train.shape[2] * 1000 / 800  # 800 ms epoch
visualize_csp_analysis(results_csp['csp_model'], results_csp['X_train_csp'], 
                      results_csp['y_train'], CHANNELS_TO_USE, sfreq_viz)


In [ ]:

# Calculate sfreq for xDawn
sfreq = X_train.shape[2] * 1000 / 800  # 800 ms epoch

print("\n--- xDawn + RandomForest ---")
results_xdawn = train_xdawn_model(X_train, X_val, X_test, y_train_raw, y_val_raw, y_test_raw,
                                  CHANNELS_TO_USE, sfreq, n_components=6, n_subsample=10000)
tracker.record('loocv', 'xDawn + RF', results_xdawn['train_bal_acc'], results_xdawn['val_bal_acc'],
               results_xdawn['test_bal_acc'], results_xdawn['train_f1'], results_xdawn['val_f1'],
               results_xdawn['test_f1'], fold_idx)


In [ ]:

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*22 + "FINAL SUMMARY" + " "*33 + "█")
print("█" + " "*68 + "█")
print("█"*70)

tracker.print_summary('normal')
tracker.print_summary('loocv')

# Compare methods
print("\n" + "="*70)
print("COMPARISON: NORMAL SPLIT vs LEAVE-N-OUT CV")
print("="*70)

df_normal = tracker.get_dataframe('normal')
df_loocv = tracker.get_dataframe('loocv')

if len(df_normal) > 0 and len(df_loocv) > 0:
    print(f"\nNormal Split:")
    print(f"  Avg Test Balanced Accuracy: {df_normal['Test_BalAcc'].mean():.1%}")
    print(f"  Avg Test F1 Macro: {df_normal['Test_F1'].mean():.1%}")
    
    print(f"\nLeave-N-Out CV:")
    print(f"  Avg Test Balanced Accuracy: {df_loocv['Test_BalAcc'].mean():.1%}")
    print(f"  Avg Test F1 Macro: {df_loocv['Test_F1'].mean():.1%}")
    
    diff_bal = df_normal['Test_BalAcc'].mean() - df_loocv['Test_BalAcc'].mean()
    if diff_bal > 0.05:
        print(f"\n⚠️  Normal split shows {diff_bal:.1%} higher balanced accuracy (possible overfitting)")
    else:
        print(f"\n✅ Similar performance across both evaluation methods")